# Task 3.2 — Failure Mode Analysis

## Failure Scenario: Non-characteristic Embedding Kernel (γ → 0)

**Description of the failure:**  
When the RBF embedding kernel bandwidth γ is set too small (γ → 0), the kernel k(x,z) = exp(-γ/2 · ‖x-z‖²) → 1 (a constant function). This violates **Assumption 1** from Task 1.2: the kernel is no longer characteristic, and the mean map μ: P → H is no longer injective. Concretely, for any two distributions P ≠ Q, μ_P ≈ μ_Q ≈ 1, so K(P_i, P_j) ≈ 1 for all pairs regardless of class. The kernel matrix becomes nearly constant, and the SVM cannot find a separating hyperplane.

**Why we expect the method to struggle:** Per Section 3 of the paper, the Gaussian RBF kernel is shown to be universal w.r.t. C(P) *given that X is compact and μ is injective* [ref 14]. When γ is too small, this injectivity collapses, and the SMM loses its representational capacity entirely.


In [ ]:
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import sys; sys.path.insert(0, '.')
from smm_core import *

np.random.seed(42); rng = np.random.RandomState(42)
N_PER_CLASS=60; N_SAMPLES=20; D=2
distributions, labels = [], []
for _ in range(N_PER_CLASS):
    distributions.append(rng.multivariate_normal(rng.randn(D)*0.6, np.array([[1.8,0.05],[0.05,0.25]]), N_SAMPLES))
    labels.append(+1)
for _ in range(N_PER_CLASS):
    distributions.append(rng.multivariate_normal(rng.randn(D)*0.6, np.array([[0.25,0.05],[0.05,1.8]]), N_SAMPLES))
    labels.append(-1)
labels = np.array(labels)
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

gammas = [0.0001, 0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 20.0]
acc_means, acc_stds = [], []

print(f"{'gamma':>10}  {'Accuracy':>12}")
print("-" * 25)
for gamma in gammas:
    scores = []
    for train_idx, test_idx in kf.split(np.arange(len(distributions)), labels):
        td = [distributions[i] for i in train_idx]
        ted = [distributions[i] for i in test_idx]
        Ktr = build_smm_kernel_matrix(td, gamma=gamma)
        Kte = build_smm_kernel_matrix_train_test(ted, td, gamma=gamma)
        clf = SVC(kernel='precomputed', C=1.0)
        clf.fit(Ktr, labels[train_idx])
        scores.append(accuracy_score(labels[test_idx], clf.predict(Kte)))
    acc_means.append(np.mean(scores)*100)
    acc_stds.append(np.std(scores)*100)
    print(f"{gamma:>10.4f}   {np.mean(scores)*100:.1f}% ± {np.std(scores)*100:.1f}%")

     gamma      Accuracy
-------------------------
    0.0001   49.2% ± 6.2%
    0.0010   50.8% ± 7.1%
    0.0100   54.2% ± 5.5%
    0.1000   75.0% ± 8.3%
    0.5000   90.8% ± 4.6%
    1.0000   89.2% ± 5.8%
    5.0000   85.0% ± 6.5%
   20.0000   82.5% ± 7.0%


## Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.errorbar(gammas, acc_means, yerr=acc_stds, fmt='o-', color='steelblue', lw=2.5, ms=8, capsize=5)
ax.axhline(50, color='red', linestyle='--', lw=1.5, label='Chance (50%)')
ax.axhline(max(acc_means), color='green', linestyle='--', lw=1.5, alpha=0.7, label=f'Best ({max(acc_means):.1f}%)')
ax.fill_between([0.00001, 0.01], 0, 100, alpha=0.12, color='red', label='Failure zone (γ < 0.01)')
ax.set_xscale('log')
ax.set_xlabel('Embedding kernel bandwidth γ', fontsize=12)
ax.set_ylabel('SMM Accuracy (%)', fontsize=12)
ax.set_ylim(0, 105)
ax.set_title('Failure Mode: γ → 0 Violates Kernel Injectivity\n(Assumption 1 from Task 1.2)', fontsize=11, fontweight='bold')
ax.legend(fontsize=10); ax.grid(alpha=0.35)
ax.annotate('K(P_i,P_j) → 1 for all i,j\nkernel loses discriminative power',
            xy=(0.001, acc_means[1]), xytext=(0.05, 30), fontsize=9.5, color='red',
            arrowprops=dict(arrowstyle='->', color='red'))

n_viz = 15
viz_dists = distributions[:n_viz] + distributions[N_PER_CLASS:N_PER_CLASS+n_viz]
ax2 = axes[1]
for gamma, col, lbl in [(0.0001, 'red', 'γ=0.0001 (flat — FAILS)'), (0.5, 'steelblue', 'γ=0.5 (discriminative — WORKS)')]:
    K = build_smm_kernel_matrix(viz_dists, gamma=gamma)
    ax2.plot(range(2*n_viz), K[0,:], color=col, lw=1.8, alpha=0.8, label=lbl)
ax2.axvline(n_viz-0.5, color='gray', linestyle='--', lw=1, label='Class boundary')
ax2.set_xlabel('Distribution index j', fontsize=12)
ax2.set_ylabel('K(P₀, P_j)', fontsize=12)
ax2.set_title('Kernel Row Profile K(P₀, P_j)\n(flat = no class separation possible)', fontsize=11, fontweight='bold')
ax2.legend(fontsize=10); ax2.grid(alpha=0.35)

plt.tight_layout()
plt.savefig('partB/results/task3_failure_mode.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: partB/results/task3_failure_mode.png")

Saved: partB/results/task3_failure_mode.png


## Explanation of Failure

The method fails catastrophically (accuracy ~50%, i.e., chance) when γ ≤ 0.001 because the RBF embedding kernel k(x,z) = exp(-γ/2·‖x-z‖²) approaches a constant function as γ → 0. When k is nearly constant, the mean embedding μ_P = ∫ k(x,·) dP(x) ≈ ∫ 1 dP(x) = 1 for any distribution P — the map becomes essentially constant and fails to be injective (violating **Assumption 1** from Task 1.2). As a result, all distributions map to nearly the same point in the RKHS, making K(P_i, P_j) ≈ 1 for all pairs. The SMM's kernel matrix is then nearly rank-1 (all entries equal), providing no discriminative signal to the SVM optimizer. This failure is directly traceable to the theoretical guarantee in Section 3 of the paper: the universal approximation property of the SMM requires a characteristic (injective) kernel; a degenerate γ breaks this guarantee.

**Suggested modification:** Automatically select γ via cross-validation as recommended in Section 6.1 (using a grid γ ∈ {10⁻³, ..., 10²}), or normalize the kernel matrix K_emp so that K(P,P) = 1 for all P, reducing sensitivity to γ scale. The paper itself uses this normalized form (NRBF) in some experiments (Table 2).
